# 🌐 06 — Prototype Aplikasi Web (Streamlit)
**Skripsi:** Sistem Rekomendasi Dosen Pembimbing Berbasis NLP — Teknik Informatika Unila

---

Notebook ini membangun dan menjalankan prototype aplikasi web menggunakan **Streamlit**
langsung dari Google Colab menggunakan tunnel `ngrok` atau `localtunnel`.

### Fitur Aplikasi
- Input judul skripsi
- Pilih metode: TF-IDF, SBERT, IndoBERT, atau SetFit
- Tampilkan Top-K rekomendasi dosen dengan skor
- Visualisasi bar chart skor per dosen

> ⚠️ **Pastikan semua notebook 01–05 sudah selesai** sebelum menjalankan ini.
> Tidak perlu GPU untuk notebook ini.

---
## 🔧 LANGKAH 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT_NAME = 'skripsi-rekomendasi-dosen'
ROOT = f'/content/drive/MyDrive/{PROJECT_NAME}'
sys.path.insert(0, os.path.join(ROOT, 'src'))
print('✅ Drive ter-mount.')

In [ ]:
!pip install -q streamlit pyngrok sentence-transformers
print('✅ Streamlit & ngrok siap.')

---
## 📌 LANGKAH 1 — Tulis File Aplikasi Streamlit

In [ ]:
import config

# Path file app.py di dalam Drive
APP_DIR  = os.path.join(ROOT, 'app')
APP_PATH = os.path.join(APP_DIR, 'app.py')
os.makedirs(APP_DIR, exist_ok=True)

app_code = f'''
# ============================================================
# app.py — Sistem Rekomendasi Dosen Pembimbing Skripsi
# Teknik Informatika — Universitas Lampung
# ============================================================

import streamlit as st
import pandas as pd
import numpy as np
import pickle, os, sys, re
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

# ─── PATH ────────────────────────────────────────────────────────
ROOT         = "{ROOT}"
DATA_PROC    = os.path.join(ROOT, "data", "processed")
MODELS_DIR   = os.path.join(ROOT, "models")

# ─── KONFIGURASI HALAMAN ─────────────────────────────────────────
st.set_page_config(
    page_title="Rekomendasi Dosen Pembimbing — TI Unila",
    page_icon="🎓",
    layout="centered",
    initial_sidebar_state="expanded",
)

# ─── CSS TAMBAHAN ────────────────────────────────────────────────
st.markdown("""
<style>
    .main-title   {{ font-size:28px; font-weight:700; color:#1565C0; margin-bottom:4px; }}
    .sub-title    {{ font-size:14px; color:#555; margin-bottom:24px; }}
    .reko-card    {{ background:#F0F7FF; border-left:4px solid #1565C0;
                    padding:12px 16px; border-radius:6px; margin:8px 0; }}
    .reko-rank    {{ font-size:22px; font-weight:700; color:#1565C0; }}
    .reko-nama    {{ font-size:16px; font-weight:600; color:#212121; }}
    .reko-skor    {{ font-size:13px; color:#666; }}
    .badge-green  {{ background:#E8F5E9; color:#2E7D32; padding:2px 10px;
                    border-radius:12px; font-size:12px; font-weight:600; }}
    .badge-blue   {{ background:#E3F2FD; color:#1565C0; padding:2px 10px;
                    border-radius:12px; font-size:12px; font-weight:600; }}
</style>
""", unsafe_allow_html=True)

# ─── LOAD MODEL & DATA (cached) ──────────────────────────────────
@st.cache_resource(show_spinner="⏳ Memuat model, harap tunggu...")
def load_all():
    import pickle, numpy as np
    from sentence_transformers import SentenceTransformer

    # Data dosen
    df_dosen = pd.read_csv(os.path.join(DATA_PROC, "profil_dosen_clean.csv"))
    df_dosen["profil_bersih_bert"] = df_dosen["profil_bersih_bert"].fillna("")
    nama_dosen   = df_dosen["nama_dosen"].tolist()
    bidang_dosen = df_dosen.get("bidang", pd.Series([""] * len(df_dosen))).tolist()

    # TF-IDF
    with open(os.path.join(MODELS_DIR, "tfidf_model.pkl"), "rb") as f:
        tfidf_obj = pickle.load(f)
    vectorizer  = tfidf_obj["vectorizer"]
    tfidf_dosen = tfidf_obj["tfidf_dosen"]

    # SBERT embedding
    emb_sbert = np.load(os.path.join(MODELS_DIR, "embeddings_dosen_sbert.npy"))

    # IndoBERT embedding
    emb_ib    = np.load(os.path.join(MODELS_DIR, "embeddings_dosen.npy"))

    # SetFit embedding (jika ada)
    setfit_path = os.path.join(MODELS_DIR, "embeddings_dosen_setfit.npy")
    emb_setfit  = np.load(setfit_path) if os.path.exists(setfit_path) else None

    # SBERT model untuk encode query
    sbert_model = SentenceTransformer("firqaaa/indo-sentence-bert-base")

    # SetFit model untuk encode query (jika ada)
    setfit_model = None
    setfit_model_path = os.path.join(MODELS_DIR, "setfit_model")
    if os.path.exists(setfit_model_path):
        from setfit import SetFitModel
        setfit_model = SetFitModel.from_pretrained(setfit_model_path).model_body

    return (nama_dosen, bidang_dosen, vectorizer, tfidf_dosen,
            emb_sbert, emb_ib, emb_setfit, sbert_model, setfit_model)

(
    NAMA_DOSEN, BIDANG_DOSEN, vectorizer, tfidf_dosen,
    emb_sbert, emb_ib, emb_setfit, sbert_model, setfit_model
) = load_all()

# ─── PREPROCESSING RINGAN (untuk query user) ─────────────────────
def clean_query(text):
    text = text.lower()
    text = re.sub(r"[^a-z\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

# ─── FUNGSI REKOMENDASI ───────────────────────────────────────────
def rekomendasikan(query_raw, metode, top_k=5):
    query_clean = clean_query(query_raw)

    if metode == "TF-IDF + Cosine Similarity":
        vec    = vectorizer.transform([query_clean])
        scores = cosine_similarity(vec, tfidf_dosen).flatten()

    elif metode == "Indo Sentence-BERT":
        q_emb  = sbert_model.encode([query_raw], normalize_embeddings=True)
        scores = cosine_similarity(q_emb, emb_sbert).flatten()

    elif metode == "IndoBERT (Mean Pooling)":
        q_emb  = sbert_model.encode([query_raw], normalize_embeddings=True)  # fallback ke SBERT
        scores = cosine_similarity(q_emb, emb_ib).flatten()

    elif metode == "SetFit Fine-tuning" and setfit_model and emb_setfit is not None:
        q_emb  = setfit_model.encode([query_raw], normalize_embeddings=True)
        scores = cosine_similarity(q_emb, emb_setfit).flatten()
    else:
        st.warning("Model SetFit tidak ditemukan. Menggunakan SBERT sebagai fallback.")
        q_emb  = sbert_model.encode([query_raw], normalize_embeddings=True)
        scores = cosine_similarity(q_emb, emb_sbert).flatten()

    ranking = np.argsort(scores)[::-1][:top_k]
    return [(NAMA_DOSEN[i], BIDANG_DOSEN[i], round(float(scores[i]), 4)) for i in ranking]

# ─── UI HEADER ───────────────────────────────────────────────────
st.markdown('<div class="main-title">🎓 Sistem Rekomendasi Dosen Pembimbing Skripsi</div>', unsafe_allow_html=True)
st.markdown('<div class="sub-title">Program Studi Teknik Informatika — Universitas Lampung</div>', unsafe_allow_html=True)
st.divider()

# ─── SIDEBAR ─────────────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ Pengaturan")

    metode_options = ["TF-IDF + Cosine Similarity", "Indo Sentence-BERT",
                      "IndoBERT (Mean Pooling)", "SetFit Fine-tuning"]
    metode_pilih   = st.selectbox("Pilih Metode:", metode_options)
    top_k          = st.slider("Jumlah Rekomendasi (K):", min_value=1, max_value=10, value=5)

    st.divider()
    st.markdown("**ℹ️ Tentang Metode:**")
    deskripsi = {{
        "TF-IDF + Cosine Similarity" : "Metode klasik berbasis frekuensi kata. Cepat dan efisien.",
        "Indo Sentence-BERT"         : "BERT khusus kalimat Indonesia. Memahami konteks semantik.",
        "IndoBERT (Mean Pooling)"     : "IndoBERT standar dengan mean pooling antar token.",
        "SetFit Fine-tuning"         : "SBERT yang di-fine-tune pada data skripsi Unila.",
    }}
    st.info(deskripsi[metode_pilih])

# ─── INPUT JUDUL ─────────────────────────────────────────────────
st.subheader("📝 Masukkan Judul atau Topik Skripsi")
judul_input = st.text_area(
    label="",
    placeholder="Contoh: Implementasi Deep Learning untuk Deteksi Penyakit Tanaman pada Citra Digital...",
    height=100,
)

col1, col2 = st.columns([1, 3])
with col1:
    cari = st.button("🔍 Cari Dosen", use_container_width=True, type="primary")
with col2:
    if judul_input:
        st.caption(f"📏 {len(judul_input.split())} kata")

# ─── HASIL REKOMENDASI ───────────────────────────────────────────
if cari:
    if not judul_input.strip():
        st.warning("⚠️ Masukkan judul skripsi terlebih dahulu.")
    else:
        with st.spinner("⏳ Menghitung kemiripan..."):
            hasil = rekomendasikan(judul_input.strip(), metode_pilih, top_k)

        st.divider()
        st.subheader(f"📋 Top-{{top_k}} Rekomendasi Dosen Pembimbing")
        st.caption(f"Metode: **{{metode_pilih}}** | Query: *{{judul_input[:60]}}...*")

        # Kartu rekomendasi
        for rank, (nama, bidang, skor) in enumerate(hasil, 1):
            skor_pct = skor * 100
            badge = "badge-green" if rank == 1 else "badge-blue"
            badge_label = "✅ Rekomendasi Utama" if rank == 1 else f"#{rank}"
            bidang_str  = f" — {bidang}" if bidang else ""

            st.markdown(f"""
            <div class="reko-card">
                <span class="reko-rank">#{rank}</span>
                <span class="badge-{"green" if rank==1 else "blue"}" style="margin-left:10px;">
                    {"✅ Utama" if rank==1 else f"#{rank}"}
                </span><br>
                <span class="reko-nama">{{nama}}</span><br>
                <span class="reko-skor">Bidang: {{bidang if bidang else "—"}}{{bidang_str}}</span><br>
                <span class="reko-skor">Skor Kemiripan: <strong>{{skor:.4f}}</strong> ({{skor_pct:.1f}}%)</span>
            </div>
            """, unsafe_allow_html=True)

        # Bar chart skor
        st.divider()
        st.subheader("📊 Visualisasi Skor Kemiripan")
        nama_list  = [f"{i+1}. {{n.split(",")[0]}}" for i, (n,b,s) in enumerate(hasil)]
        skor_list  = [s for _,_,s in hasil]
        warna_list = ["#1565C0" if i == 0 else "#90CAF9" for i in range(len(hasil))]

        fig, ax = plt.subplots(figsize=(8, max(3, len(hasil)*0.55)))
        bars = ax.barh(nama_list[::-1], skor_list[::-1], color=warna_list[::-1], edgecolor="white")
        for bar, skor in zip(bars, skor_list[::-1]):
            ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
                    f"{{skor:.4f}}", va="center", fontsize=9, fontweight="bold")
        ax.set_xlim(0, max(skor_list)*1.25)
        ax.set_xlabel("Cosine Similarity Score", fontsize=10)
        ax.set_title(f"Skor Kemiripan — {{metode_pilih}}", fontweight="bold", fontsize=11)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.set_facecolor("#FAFAFA")
        st.pyplot(fig)
        plt.close()

# ─── FOOTER ──────────────────────────────────────────────────────
st.divider()
st.caption("🎓 Sistem Rekomendasi Dosen Pembimbing | Teknik Informatika Unila | Prototipe Penelitian Skripsi")
'''

with open(APP_PATH, 'w', encoding='utf-8') as f:
    f.write(app_code)

print(f'✅ app.py berhasil ditulis ke: {APP_PATH}')

---
## 📌 LANGKAH 2 — Setup ngrok & Jalankan Streamlit

In [ ]:
# ─── DAFTAR DAN MASUKKAN NGROK TOKEN ─────────────────────────────
# 1. Daftar gratis di https://ngrok.com
# 2. Salin authtoken dari dashboard ngrok
# 3. Tempelkan di sini:

NGROK_TOKEN = "MASUKKAN_NGROK_TOKEN_KAMU_DI_SINI"  # ← ganti ini

from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_TOKEN
print('✅ ngrok token terdaftar.')

In [ ]:
# ─── JALANKAN STREAMLIT + BUKA TUNNEL ────────────────────────────
import subprocess, time

PORT = 8501

# Matikan proses streamlit sebelumnya (jika ada)
!pkill -f streamlit 2>/dev/null || true
time.sleep(1)

# Jalankan Streamlit di background
proc = subprocess.Popen(
    ['streamlit', 'run', APP_PATH,
     '--server.port', str(PORT),
     '--server.headless', 'true',
     '--server.fileWatcherType', 'none'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

time.sleep(4)  # tunggu streamlit start

# Buka tunnel ngrok
public_url = ngrok.connect(PORT)

print('=' * 55)
print('🌐 APLIKASI BERHASIL DIJALANKAN!')
print('=' * 55)
print(f'\n   URL Publik: {public_url}')
print()
print('   Buka URL di atas di browser untuk mengakses aplikasi.')
print('   Aplikasi akan aktif selama sesi Colab ini berjalan.')
print()
print('   ⚠️  Jangan close tab Colab ini selama demo berlangsung!')
print('=' * 55)

In [ ]:
# ─── ALTERNATIF: Localtunnel (tanpa perlu daftar) ─────────────────
# Jika tidak ingin daftar ngrok, gunakan localtunnel sebagai alternatif.
# Jalankan cell ini SEBAGAI PENGGANTI cell ngrok di atas (bukan bersamaan).

# !npm install -q localtunnel
# !streamlit run {APP_PATH} --server.port 8501 &
# import time; time.sleep(4)
# !npx localtunnel --port 8501 &
# print('Tunggu ~5 detik, lalu lihat URL localtunnel di output di atas.')

In [ ]:
# ─── STOP APLIKASI ───────────────────────────────────────────────
# Jalankan cell ini untuk menghentikan aplikasi dan menutup tunnel

# ngrok.kill()
# proc.terminate()
# print('🛑 Aplikasi dihentikan.')

---
## ✅ Selesai — Ringkasan Notebook 06

| Output | Lokasi |
|--------|--------|
| Kode aplikasi web | `app/app.py` |
| URL publik (sementara) | Dari ngrok saat runtime aktif |

---

## 🎉 Seluruh Pipeline Selesai!

```
00_setup              ✅ Environment & struktur folder
01_load_data          ✅ Load CSV skripsi + scraping SINTA
02_preprocessing      ✅ Cleaning, stopword, stemming
02B_patch             ✅ Generate labeled pairs (Opsi B)
03_tfidf_baseline     ✅ TF-IDF + Cosine Similarity
04A_bert_embedding    ✅ IndoBERT & SBERT embedding
04B_setfit            ✅ SetFit fine-tuning (Opsi B)
05_evaluasi           ✅ Evaluasi final & visualisasi
06_prototype_app      ✅ Aplikasi web Streamlit
```

Semua hasil tersimpan di Google Drive kamu di folder `skripsi-rekomendasi-dosen/`.